In [19]:
import os
import math

import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch.optim as optim

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

In [29]:
#Some constants/Hyperparameters
VOCAB_SIZE = 3000              #How many tokens to create

BATCH_SIZE = 32
CONTEXT_WINDOW = 128
D_MODEL = 256

NUM_HEADS = 4
FFN_HIDDEN = 512
NUM_BLOCKS = 3

Pulling the Raw Data

Training Custom Tokenizer and Tokenizing Data

In [5]:
#This is the tokenization cell
#TrainTokenizer uses HuggingFaces tokenizers library to create it's own token/vocab set because or data set is
#   rather small with only ~890,000 words
def TrainTokenizer(raw_data, save_name):
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=VOCAB_SIZE, 
        special_tokens=["<unk>", "<s>", "</s>"]
    )

    temp_file = "temp_train_data.txt"
    with open(temp_file, "w", encoding="utf-8") as f:
        f.write(raw_data)
    
    tokenizer.train([temp_file], trainer)

    os.remove(temp_file)

    os.makedirs("data/tokenized", exist_ok=True)
    save_path = f"data/tokenized/{save_name}.json"
    tokenizer.save(save_path)

    print(f"Created tokenized dataset with {VOCAB_SIZE} tokens and saved to {save_path}")
    return tokenizer

#This uses the tokenizer we trained and actually tokenized the dataset so that now it's just a long list of
#   numbers/tokens 
def Tokenize(raw_data, save_name):
    my_tokenizer = TrainTokenizer(raw_data, save_name)
    tokenized_data = my_tokenizer.encode(raw_data).ids
    print("Tokenized all data")
    return tokenized_data

Creating Training Batches with Dataset & DataLoaders

In [37]:
#This cell is where we create our training batches
#We use Pytorch dataset to act as a plucking tool to pluck out a single training/target
#   tensors from the dataset
class PTBTrain(Dataset):
    def __init__(self, token_list, context_window):
        self.token_list = token_list
        self.context_window = context_window
    
    def __len__(self):
        return len(self.token_list) - self.context_window

    def __getitem__(self, idx):
        x = torch.tensor(self.token_list[idx:idx + self.context_window], dtype=torch.long)
        y = torch.tensor(self.token_list[idx + 1:idx + self.context_window + 1], dtype=torch.long)

        return x, y

Embeddings Layer

In [9]:
#This is the embedding layer and what actually gets fed into the model
#This is the simpler version for now with just regular plain embeddings
#   but I will soon add the RoPE math and everything
class EmbeddingLayer(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.token_embedding = nn.Embedding(vocab_size, d_model)
    
    def forward(self, batch):
        return self.token_embedding(batch)

Rotary Positional Encodings (RoPE)

In [11]:
def RoPE(d_model, num_heads, context_window):
    mat_dim = int(d_model/num_heads)
    base = 10000
    rotation_matrix = torch.zeros(context_window, mat_dim, mat_dim)

    thetas = 1 / torch.pow(base, (2 * torch.arange(0, mat_dim/2, 1) / mat_dim))
    theta_m = torch.outer(torch.arange(0, context_window, 1), thetas)

    for m in range(0, context_window):
        for i in range(0, int(mat_dim/2)):
            cos_m = torch.cos(theta_m[m,i])
            sin_m = torch.sin(theta_m[m,i])

            rotation_matrix[m, 2*i, 2*i] = cos_m
            rotation_matrix[m, 2*i, 2*i+1] = -sin_m
            rotation_matrix[m, 2*i+1, 2*i] = sin_m
            rotation_matrix[m, 2*i+1, 2*i+1] = cos_m
    
    return rotation_matrix

Feed Forward Neural Net (Decoder Block)

In [ ]:
#Creating the actual Transfomer Architecture now
#We will do a single hidden layer for now
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)
        self.GeLU = nn.GELU()
    
    def forward(self, batch):
        return self.w2(self.GeLU(self.w1(batch)))

Single Attention Head

In [16]:
#Creating a single attention head right now
#Not the best practice but we will adjust/update later on
#Shape: (batch_size, context_window, d_model) ---> (batch_size, context_window, d_model/num_heads)
class AttentionHead(nn.Module):
    def __init__(self, d_model, num_heads, context_window):
        super().__init__()
        #Projection dim for emb_dim --> qkv dim
        self.qkv_dim = int(d_model/num_heads)

        #Intialize the QKV Matrices
        self.query = nn.Linear(d_model, self.qkv_dim, bias=False)
        self.key = nn.Linear(d_model, self.qkv_dim, bias=False)
        self.value = nn.Linear(d_model, self.qkv_dim, bias=False)

        #Creating the RoPE Matrix (not the most efficient way as of now)
        self.rotation_mat = RoPE(d_model, num_heads, context_window)
        
        #Creating the mask for the attention head, we are storing as attribute
        #Finds the upper triangle and sets it to ones --> bools
        self.mask = torch.triu(torch.ones(context_window, context_window), diagonal=1).bool()
    
    def forward(self, batch):
        #Actual Q @ K.T in attention, the transpose just swaps the last two dims in the shape
        #RoPE has Shape: (context_window, qkv_dim, qkv_dim)
        #Shape: (batch_size, context_window, qkv_dim)

        query = self.rotation_mat.unsqueeze(0) @ self.query(batch).unsqueeze(-1)
        key = self.rotation_mat.unsqueeze(0) @ self.key(batch).unsqueeze(-1)

        qk = query.squeeze(-1) @ key.squeeze(-1).mT
        attention_score_logits = qk / math.sqrt(self.qkv_dim)
        
        #Actual attention score probabilites
        #Shape: (batch_size, context_window, context_window)
        attention_score_logits.masked_fill_(self.mask, value=float("-inf"))
        attention_scores = attention_score_logits.softmax(dim=-1)

        #Shape: (batch_size, context_window, qkv_dim)
        attention_embeddings = attention_scores @ self.value(batch)

        return attention_embeddings

Multi Headed Attention

In [ ]:
#Class for all the attention heads
#Shape: list of length num_heads (batch_size, context_window, d_model/num_heads) ---> (batch_size, context_window, d_model)
class MultiHeadedAttention(nn.Module):
    def __init__(self, d_model, num_heads, context_window):
        super().__init__()
        self.num_heads = num_heads
        self.attention_heads = nn.ModuleList([
            AttentionHead(d_model, num_heads, context_window)
            for i in range(num_heads)
        ])

        #Mixing all the information from the attention heads
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
    
    def forward(self, batch):
        #A list of the outputs of each attention head
        #Shape: (batch_size, context_window, d_model/num_heads)
        forward_head = [head(batch) for head in self.attention_heads]

        #Shape: (batch_size, context_window, d_model)
        output = torch.cat(forward_head, dim=-1)
        return self.out_proj(output)


Decoder Block

In [21]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, context_window, d_ff):
        super().__init__()
        self.num_heads = num_heads

        self.multi_headed_attention = MultiHeadedAttention(
            d_model,
            num_heads,
            context_window
        )
        
        self.ffn = FeedForward(
            d_model, 
            d_ff
        )

        self.pre_RMSNorm = nn.RMSNorm(d_model)
        self.post_RMSNorm = nn.RMSNorm(d_model)

    def forward(self, X):
        #Shape: (batch_size, context_window, d_model)
        #Normalize X (Pre RMSNorm) ---> Attention ---> X + Normalize Attention Ouput (Post RMSNorm)
        X_norm = self.pre_RMSNorm(X)
        mha = self.multi_headed_attention(X_norm)

        #Make sure we don't forget the original embeddings (pre-attention)
        X_mid = X + mha
        
        #Normalize Attention Output ---> FFN ---> Final Decoder Block Output
        ffn_in = self.post_RMSNorm(X_mid)
        ffn = self.ffn(ffn_in)

        #Make sure we don't lose information after FFN
        output = X_mid + ffn
        return output


Transformer Architecture (Putting Everything Together)

In [22]:
#Full walk through of Class
#   Pre-class we tokenize all data and set up the dataloaders
#       - Shape we are feeding into embedding layer |   Shape: (batch_size, context_window)
#   Embedding Layer (Pre-attention, non-context rich embeddings)    |   Shape: (batch_size, context_window, d_model)
#   3 Decoder Blocks
#   Apply RMSNorm one last time
#   LM Head to go from (batch_size, context_window, d_model) ---> (batch_size, context_window, token_count)
#       - This is our predicitions for each
#   Softmax layer and get our predicitions
class Transformer(nn.Module):
    def __init__(self, num_blocks, d_model, vocab_size, num_heads, d_ff, context_window):
        super().__init__()
        self.embedding_layer = EmbeddingLayer(vocab_size, d_model)
        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, context_window, d_ff)
            for _ in range(num_blocks)
        ])

        self.RMSNorm = nn.RMSNorm(d_model)
        self.LM_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, batch):
        X = self.embedding_layer(batch)

        for block in self.blocks:
            X = block(X)
        
        X = self.RMSNorm(X)
        logits = self.LM_head(X)
        return logits

Training Script

In [52]:
# 1. Pull the raw data from our files
train_raw = open('data/ptb.train.txt', 'r').read()
test_raw = open('data/ptb.test.txt', 'r').read()
val_raw = open('data/ptb.val.txt', 'r').read()

# 2. Tokenize the train/validation data at once
train_tokens = Tokenize(train_raw, "ptb.train_tokens.txt")
val_tokens = Tokenize(val_raw, "ptb.val_token.txt")

# 3. Setting up Data pulling and batching
#   ptb_dataset pulls a single sample of tokens from our tokenized data
train_data = PTBTrain(train_tokens, CONTEXT_WINDOW)
val_data = PTBTrain(val_tokens, CONTEXT_WINDOW)

#Actually sets up the training/label batches which we feed into Transformer and eventually calculate loss on
training_batches = DataLoader(
    dataset=train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

#Setting up the validation batches to ensure the model isn't just overfitting and memorizing our training data
val_batches = DataLoader(
    dataset=val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=True
)

#How many training and validation batches we actually have
print(len(training_batches))
print(len(val_batches))


# 4. Intializing transformer
model = Transformer(NUM_BLOCKS, D_MODEL, VOCAB_SIZE, NUM_HEADS, FFN_HIDDEN, CONTEXT_WINDOW)

# # 5. Train transformer
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001, betas=(0.9, 0.999))

# train_loss = []
# val_loss = []
# epochs = 10

# for epoch in range(epochs):
#     total_train_loss = 0
#     model.train()
#     for batch_x, batch_y in training_batches:
        
#         #Forward Pass through entire transformer
#         #Shape: (batch_size, context_window, vocab_size)
#         logits = model(batch_x)

#         #Takes in Shape: (batch_size * context_window, d_model) |   (batch_size * context_window)
#         #We don't care if the prediction came from batch 2 or 30 as long as we keep the context_window in order
#         loss = criterion(logits.view(-1, VOCAB_SIZE), batch_y.view(-1))
#         train_loss.append(loss.item())
#         total_train_loss += loss.item()

#         #Set up optimizer to update parameters based on gradients
#         optimizer.zero_grad()   #Removes the gradients and sets it to 0, otherwise we would accumulate gradients
#         loss.backward()
#         optimizer.step()

#     total_val_loss = 0
#     best_val_loss = float("inf")
#     model.eval()
#     with torch.no_grad():
#         for val_x, val_y in val_batches:
#             logits = model(val_x)

#             loss = criterion(logits.view(-1, VOCAB_SIZE), val_y.view(-1))
#             val_loss.append(loss.item())
#             total_val_loss += loss.item()

#     avg_train_loss = total_train_loss / len(training_batches)
#     avg_val_loss = total_val_loss / len(val_batches)

#     #Saving the weights where validation loss is the best
#     if avg_val_loss < best_val_loss:
#         torch.save(model.state_dict(), "data/weights/best_model.pt")
#         best_val_loss = avg_val_loss
#         print("Saved Weights")

#     #Standard measurement for Language Models
#     #Measures how confident a model is in it's predicition
#     perplexity = math.exp(avg_val_loss)
#     print(f"Epoch: {epoch}\nTraining Loss | {avg_train_loss} Validation Loss {avg_val_loss} | Perplexity: {perplexity}")


Created tokenized dataset with 3000 tokens and saved to data/tokenized/ptb.train_tokens.txt.json
Tokenized all data
Created tokenized dataset with 3000 tokens and saved to data/tokenized/ptb.val_token.txt.json
Tokenized all data
35730
2713


Validation/Model Expirements & Learning

In [44]:
#Counting total number of parameters
total_params = sum(
    p.numel() for p in model.parameters()
)

print(total_params)

3112960


In [50]:
# A good practice is to take a very small sample (say 1000 tokens) and see if the model can overfit on the data
# This means the model is actually learning and it's working
sample = train_tokens[:3000]

tiny_dataset = PTBTrain(
    sample, 
    CONTEXT_WINDOW
)

tiny_training_batches = DataLoader(
    dataset=tiny_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)

tiny_model = Transformer(NUM_BLOCKS, D_MODEL, VOCAB_SIZE, NUM_HEADS, FFN_HIDDEN, CONTEXT_WINDOW)
tiny_optimizer = optim.Adam(tiny_model.parameters(), lr=0.001, betas=(0.9, 0.999))
tiny_criterion = nn.CrossEntropyLoss()

tiny_step = 0
epochs = 2
final_loss = 0
for epoch in range(epochs):
    model.train()
    tiny_step = 0
    for X, Y in tiny_training_batches:
        logits = tiny_model(X)
        loss = tiny_criterion(logits.view(-1, VOCAB_SIZE), Y.view(-1))
        tiny_optimizer.zero_grad()
        loss.backward()
        tiny_optimizer.step()
        
        final_loss = loss.item()
        if tiny_step % 50 == 0:
            print(f"Epoch {epoch} | Step {tiny_step} | Loss {loss.item():.4f}")
        tiny_step += 1
print(f"Loss after {epochs} epochs: {final_loss:.4f}")


Epoch 0 | Step 0 | Loss 8.1585
Epoch 0 | Step 50 | Loss 1.1593
Epoch 1 | Step 0 | Loss 0.1473
Epoch 1 | Step 50 | Loss 0.0718
Loss after 2 epochs: 0.0485
